# 03 – Privacy Demo (Pseudonymization)

This notebook demonstrates the identification of personal data (PII) in the credit application dataset and applies basic pseudonymization techniques to reduce privacy risks while preserving analytical usefulness.

In [1]:
import json
import pandas as pd
import hashlib
import uuid
from pathlib import Path

In [3]:
from pathlib import Path
import json
import pandas as pd

DATA_PATH = Path("../data/raw/raw_credit_applications.json")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.json_normalize(raw_data)
df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


## Identified Personal Data (PII)

Direct identifiers:
- applicant_info.full_name
- applicant_info.email
- applicant_info.ssn

Personal data / quasi-identifiers:
- applicant_info.ip_address
- applicant_info.date_of_birth
- applicant_info.zip_code

In [4]:
def hash_email(email, salt="nova_cred_salt"):
    if pd.isna(email):
        return email
    return hashlib.sha256((str(email) + salt).encode("utf-8")).hexdigest()

df["email_hashed"] = df["applicant_info.email"].apply(hash_email)

df[["applicant_info.email", "email_hashed"]].head()

,applicant_info.email,email_hashed
0,jerry.smith17@hotmail.com,546945df458fade3935f143b07bd127c9b7bca1d49b5db...
1,brandon.walker2@yahoo.com,1f6ecebcf9aef44e8511d98c10d08bc31d1dbb4029f8b7...
2,scott.moore94@mail.com,e05544796e639ee6b70320e170b73bef29e14bdf9fd061...
3,thomas.lee6@protonmail.com,8c1079dcbdfd13e46cda2f70cda34fe4186e2d4292b187...
4,brian.rodriguez86@aol.com,056df7d69cf1c63c6dca2f8712ef748ece544f9d984d35...


In [5]:
token_map = {}

def tokenize_ssn(ssn):
    if pd.isna(ssn):
        return ssn
    ssn = str(ssn)
    if ssn not in token_map:
        token_map[ssn] = str(uuid.uuid4())
    return token_map[ssn]

df["ssn_token"] = df["applicant_info.ssn"].apply(tokenize_ssn)

df[["applicant_info.ssn", "ssn_token"]].head()

,applicant_info.ssn,ssn_token
0,596-64-4340,00e6a899-1766-4a21-b82e-a83823c92df4
1,425-69-4784,1e8aabf6-d8c9-402c-b882-289969f48ffd
2,370-78-5178,4253f9b8-6ce5-4832-9f26-e180e4505979
3,194-35-1833,e8b63ae5-b284-46e5-a199-741b74b06c5d
4,480-41-2475,5ebd2517-16bc-4d79-beea-2caaa9caf304


In [6]:
sanitized_df = df.drop(columns=[
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.full_name"
])

sanitized_df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,email_hashed,ssn_token
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,546945df458fade3935f143b07bd127c9b7bca1d49b5db...,00e6a899-1766-4a21-b82e-a83823c92df4
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,1f6ecebcf9aef44e8511d98c10d08bc31d1dbb4029f8b7...,1e8aabf6-d8c9-402c-b882-289969f48ffd
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,e05544796e639ee6b70320e170b73bef29e14bdf9fd061...,4253f9b8-6ce5-4832-9f26-e180e4505979
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,192.168.175.67,Male,1983-04-25,10077,103000,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,8c1079dcbdfd13e46cda2f70cda34fe4186e2d4292b187...,e8b63ae5-b284-46e5-a199-741b74b06c5d
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,172.29.125.105,M,1999-05-21,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,056df7d69cf1c63c6dca2f8712ef748ece544f9d984d35...,5ebd2517-16bc-4d79-beea-2caaa9caf304
